# 05 — Model Training and Evaluation

Compare Logistic Regression (interpretable baseline) and Random Forest (non-linear candidate). Model selection uses validation PR-AUC. A final model is refit on train + validation and evaluated once on a later held-out test period.

A 90-day embargo is used between temporal partitions to reduce overlap between forward-looking labels and later periods. Metrics are only available after running the notebook on the actual dataset.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src" / "train.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.features import FEATURES_FILE
from src.train import chronological_split, main as train_model, REPORT_DIR

if not FEATURES_FILE.exists():
    raise FileNotFoundError("Run Notebook 04 first to create customer_snapshots.parquet.")
data = pd.read_parquet(FEATURES_FILE)
data["snapshot_date"] = pd.to_datetime(data["snapshot_date"])
print("Rows:", len(data), "| Distinct snapshot months:", data.snapshot_date.nunique())

Rows: 48079 | Distinct snapshot months: 16


## 1. Inspect chronological partitions

In [2]:
train, validation, test = chronological_split(data)
for name, part in [("train", train), ("validation", validation), ("test", test)]:
    print(name, len(part), part.snapshot_date.min(), "to", part.snapshot_date.max(),
          "| positive rate:", round(part.repeat_purchase_90d.mean(), 4))

train 20677 2010-06-01 00:00:00 to 2010-12-01 00:00:00 | positive rate: 0.5162
validation 6653 2011-03-01 00:00:00 to 2011-04-01 00:00:00 | positive rate: 0.4311
test 8267 2011-07-01 00:00:00 to 2011-09-01 00:00:00 | positive rate: 0.5611


## 2. Train, select and evaluate

The training module fits the two candidate pipelines, selects by validation PR-AUC, refits the selected model on development data, and evaluates on the untouched test period.

In [3]:
train_model()
metrics_path = REPORT_DIR / "model_metrics.json"
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print("Selected model:", metrics["selected_model"])
test_metrics = metrics["test_metrics"].copy()
confusion_matrix = test_metrics.pop("confusion_matrix")

display(pd.Series(test_metrics, name="held_out_test").to_frame())
display(pd.DataFrame(
    confusion_matrix,
    index=["actual_0", "actual_1"],
    columns=["predicted_0", "predicted_1"],
))
display(pd.DataFrame(metrics["capacity_metrics"]).T)

{
  "target": "repeat_purchase_90d",
  "definition": "At least one valid purchase in the 90 days after snapshot date",
  "selected_model": "random_forest",
  "train_rows": 20677,
  "validation_rows": 6653,
  "test_rows": 8267,
  "train_period": [
    "2010-06-01",
    "2010-12-01"
  ],
  "validation_period": [
    "2011-03-01",
    "2011-04-01"
  ],
  "test_period": [
    "2011-07-01",
    "2011-09-01"
  ],
  "label_horizon_embargo_days": 90,
  "validation_metrics": {
    "logistic_regression": {
      "precision": 0.6800165152766309,
      "recall": 0.5742677824267782,
      "f1": 0.6226843100189036,
      "confusion_matrix": [
        [
          3010,
          775
        ],
        [
          1221,
          1647
        ]
      ],
      "positive_rate": 0.43108372162934017,
      "roc_auc": 0.7447822185865443,
      "pr_auc": 0.7249618734044583
    },
    "random_forest": {
      "precision": 0.6350017024174327,
      "recall": 0.650278940027894,
      "f1": 0.6425495262704565,


,held_out_test
precision,0.779391
recall,0.590213
f1,0.671737
positive_rate,0.561147
roc_auc,0.750903
pr_auc,0.809654


,predicted_0,predicted_1
actual_0,2853,775
actual_1,1901,2738


,customers_selected,precision_at_k,lift_at_k
top_10pct,826.0,0.946731,1.687137
top_20pct,1653.0,0.897157,1.598792


## 3. Interpretation

- PR-AUC is especially useful when positive and negative classes are imbalanced.
- Precision@10% and lift@10% estimate how concentrated future repeat purchasers are among the highest-scored customers.
- A 0.50 threshold is not automatically optimal for a business outreach programme.
- Compare results with a simple baseline and inspect performance across time before making operational claims.

Do not describe the output as a churn probability or as evidence that a campaign will cause a purchase.